# 03 · 챗봇 RAG 파이프라인

**`ChatbotRAGPipeline`** 클래스를 사용합니다.

금칙어 필터 → 기사/카드 RAG 검색 → 히스토리 관리 → 응답 생성 → **CHAT_SESSIONS / CHAT_MESSAGES** RDB 저장

## 0. 셋업

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from openai import OpenAI
from config import OPENAI_API_KEY, DB_URL
from db.rdb import get_engine, init_tables
from db.vectordb import get_chroma_client
from pipelines.chatbot_rag import ChatbotRAGPipeline

client = OpenAI(api_key=OPENAI_API_KEY)
engine = get_engine(DB_URL)
chroma = get_chroma_client()
init_tables(engine)

best_file = Path("./output/best_combo.txt")
combo     = best_file.read_text().strip() if best_file.exists() else "sentence_large"
parts     = combo.rsplit("_", 1)
STRATEGY, MODEL_KEY = parts[0], parts[1]

pipeline = ChatbotRAGPipeline(
    engine=engine, chroma=chroma, openai_client=client,
    strategy=STRATEGY, model_key=MODEL_KEY,
)
print(f"ChatbotRAGPipeline 초기화 완료  (strategy={STRATEGY}, model={MODEL_KEY})")


## 1. 금칙어 필터 테스트

In [ ]:
test_cases = [
    ("청년 주거 지원 정책 알려줘",    False),
    ("이 정책은 진짜 쓰레기야",       True),
    ("XX당은 다 거짓말쟁이다",        True),
]

print("금칙어 필터 테스트")
print("-" * 50)
for text, expected in test_cases:
    blocked, reason = pipeline._check_profanity(text)
    status = "🚫 차단" if blocked else "✅ 통과"
    match  = "✓" if blocked == expected else "✗ (예상과 다름)"
    print(f"{status} {match}  |  {text[:30]}  |  {reason}")


## 2. RAG 검색 테스트

In [ ]:
from db.vectordb import retrieve_from_articles, retrieve_from_cards

query = "청년 월세 지원 신청 방법"
print(f"쿼리: '{query}'\n")

art_results  = retrieve_from_articles(query, chroma, STRATEGY, MODEL_KEY, client, top_k=3)
card_results = retrieve_from_cards(query, chroma, MODEL_KEY, client, top_k=3)

print(f"기사 검색 결과 {len(art_results)}건:")
for r in art_results:
    print(f"  score={r['score']:.4f}  {r['metadata'].get('title','')[:50]}")

print(f"\n카드 검색 결과 {len(card_results)}건:")
for r in card_results:
    print(f"  score={r['score']:.4f}  card_id={r['metadata'].get('card_id')}")


## 3. 챗봇 대화 테스트

In [ ]:
# 테스트용 사용자 ID (실제 서비스에서는 인증된 user_id 사용)
TEST_USER_ID = 1

# 세션 생성
session_id = pipeline.create_session(TEST_USER_ID, "주거 정책 문의")
print(f"세션 생성: #{session_id}\n")

# 멀티턴 대화 테스트
questions = [
    "청년 주거 지원 정책이 뭐가 있어?",
    "방금 말한 정책 신청 자격이 어떻게 돼?",
]

for q in questions:
    print(f"👤 사용자: {q}")
    reply, recs = pipeline.chat(session_id, q)
    print(f"🤖 챗봇: {reply[:250]}{'...' if len(reply)>250 else ''}")
    if recs:
        print(f"📌 추천 카드 {len(recs)}개:")
        for r in recs:
            print(f"   card #{r['card_id']} ({r['card_type']}) score={r['score']}")
    print()


## 4. 히스토리 & 세션 관리

In [ ]:
# 히스토리 확인
history = pipeline.get_history(session_id)
print(f"세션 #{session_id} 메시지 {len(history)}건:")
for msg in history:
    role_icon = "👤" if msg["role"] == "user" else "🤖"
    print(f"  {role_icon} [{msg['role']}] {msg['content'][:80]}...")

# 세션 목록 확인
sessions = pipeline.list_sessions(TEST_USER_ID)
print(f"\n사용자 #{TEST_USER_ID} 세션 목록: {len(sessions)}개")
for s in sessions:
    print(f"  #{s['id']}: {s['title']} ({s['created_at']})")


## 5. 인터랙티브 루프

In [ ]:
# 종료: "quit" 입력
CHAT_HISTORY: list[dict] = []
DEMO_SESSION = pipeline.create_session(TEST_USER_ID, "인터랙티브 데모")

print("챗봇 인터랙티브 모드 (종료: quit)\n")
while True:
    user_input = input("👤 사용자: ").strip()
    if not user_input or user_input.lower() in ("quit", "종료", "exit"):
        print("종료합니다.")
        break
    reply, recs = pipeline.chat(DEMO_SESSION, user_input)
    print(f"\n🤖 챗봇: {reply}\n")
    if recs:
        print("📌 추천 카드:")
        for r in recs:
            print(f"   card #{r['card_id']} {r['card_type']} ({r['score']:.3f})")
    print()
